In [77]:
import numpy as np
import os
import PIL
import PIL.Image
import tensorflow as tf
import cv2

In [78]:
print(tf.__version__)
print(tf.config.list_physical_devices('GPU'))

2.21.0
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [79]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("rahmasleam/flowers-dataset")

# print("Path to dataset files:", path)

# Debug imgs
# data_dir = "/home/parisibr/Atividade-Aula-Pr-tica-MLP-CNN/dataset/versions/1/flower_photos"
# img_path = r'/home/parisibr/Atividade-Aula-Pr-tica-MLP-CNN/dataset/versions/1/flower_photos/daisy/100080576_f52e8ee070_n.jpg'
# img = cv2.imread(img_path)
# height, width, channels = img.shape
# print(f"Width: {width}, Height: {height}, Channels: {channels}")

In [80]:
# Dataset root dir
data_dir = "/home/parisibr/Atividade-Aula-Pr-tica-MLP-CNN/dataset/versions/1/flower_photos"

# Definições de dataset
batch_size = 32
img_height = 263
img_width = 320


# Carrega dataset de treinamento 70%
train_ds = tf.keras.utils.image_dataset_from_directory(
  data_dir,
  validation_split=0.7,
  subset="training",
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size)

# Carrega dataset de validação 15%
val_ds = tf.keras.utils.image_dataset_from_directory(
  data_dir,
  validation_split=0.15,
  subset="validation",
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size)

class_names = train_ds.class_names
print(class_names)

Found 3670 files belonging to 5 classes.
Using 1101 files for training.
Found 3670 files belonging to 5 classes.
Using 550 files for validation.
['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']


In [81]:
# Configurar conjunto de dados para desempenho (Adicionar isso implicou no erro da visualização de dados v1 na cell abaixo, então comentei por enquanto)

# def configure_for_performance(ds):
#   ds = ds.cache()
#   ds = ds.shuffle(buffer_size=1000)
#   ds = ds.batch(batch_size)
#   ds = ds.prefetch(buffer_size=AUTOTUNE)
#   return ds

# train_ds = configure_for_performance(train_ds)
# val_ds = configure_for_performance(val_ds)

In [82]:
# Visualização dos dados v0

# import matplotlib.pyplot as plt

# plt.figure(figsize=(10, 10))
# for images, labels in train_ds.take(1):
#   for i in range(9):
#     ax = plt.subplot(3, 3, i + 1)
#     plt.imshow(images[i].numpy().astype("uint8"))
#     plt.title(class_names[labels[i]])
#     plt.axis("off")

# for image_batch, labels_batch in train_ds:
#   print(image_batch.shape) # Tensor da forma (32,263,320,3)
#   print(labels_batch.shape) # Tensor da forma (32,)
#   break

# Visualização dos dados v1 (batch issue - NÃO ESTÁ FUNCIONANDO -  Cannot batch tensors with different shapes in component)

# import matplotlib.pyplot as plt
# image_batch, label_batch = next(iter(train_ds))

# plt.figure(figsize=(10, 10))
# for i in range(9):
#   ax = plt.subplot(3, 3, i + 1)
#   plt.imshow(image_batch[i].numpy().astype("uint8"))
#   label = label_batch[i]
#   plt.title(class_names[label])
#   plt.axis("off")

In [83]:
# Padronização dos dados

# Padronizando os valores para estarem no intervalo [0, 1]
normalization_layer = tf.keras.layers.Rescaling(1./255)

normalized_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
image_batch, labels_batch = next(iter(normalized_ds))
first_image = image_batch[0]
# Notice the pixel values are now in `[0,1]`.
print(np.min(first_image), np.max(first_image))

0.0 1.0


In [84]:
# Configurar o conjunto de dados para desempenho

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [85]:
# Treinar um modelo

num_classes = 5

model = tf.keras.Sequential([
  tf.keras.layers.Rescaling(1./255),
  tf.keras.layers.Conv2D(32, 3, activation='relu'),
  tf.keras.layers.MaxPooling2D(),
  tf.keras.layers.Conv2D(32, 3, activation='relu'),
  tf.keras.layers.MaxPooling2D(),
  tf.keras.layers.Conv2D(32, 3, activation='relu'),
  tf.keras.layers.MaxPooling2D(),
  tf.keras.layers.Flatten(),
  tf.keras.layers.Dense(128, activation='relu'),
  tf.keras.layers.Dense(num_classes)
])

model.compile(
  optimizer='adam',
  loss=tf.losses.SparseCategoricalCrossentropy(from_logits=True),
  metrics=['accuracy'])

model.fit(
  train_ds,
  validation_data=val_ds,
  epochs=3
)

Epoch 1/3


I0000 00:00:1780614274.303885   58809 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_6767__.37


34/35 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step - accuracy: 0.2853 - loss: 2.2444

I0000 00:00:1780614279.401431   58808 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_6767__.37


35/35 ━━━━━━━━━━━━━━━━━━━━ 9s 184ms/step - accuracy: 0.2906 - loss: 1.7683 - val_accuracy: 0.3564 - val_loss: 1.4104
Epoch 2/3
35/35 ━━━━━━━━━━━━━━━━━━━━ 4s 108ms/step - accuracy: 0.4578 - loss: 1.2433 - val_accuracy: 0.5127 - val_loss: 1.1848
Epoch 3/3
35/35 ━━━━━━━━━━━━━━━━━━━━ 4s 101ms/step - accuracy: 0.5786 - loss: 1.0255 - val_accuracy: 0.5527 - val_loss: 1.1786
